# 03 · CD Spectroscopy: Seeing Protein Shape with Light
### *From helices and sheets to spectra — and back again*

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/elkins-lab/diff-biophys/blob/main/examples/interactive_tutorials/03_cd_spectroscopy.ipynb)

---

**Prerequisites:** Notebook 01 (gradient descent) — no NMR knowledge needed.

**What you will learn:**
1. What circular dichroism (CD) spectroscopy is and *why* it encodes secondary structure
2. How to build a protein helix from scratch using differentiable geometry (NeRF)
3. How `simulate_cd_matrix` computes a CD spectrum from atomic positions
4. How the spectrum changes as you distort the helix
5. How gradients flow from the CD spectrum back to chromophore positions

**Time:** ~40 minutes


In [ ]:
%pip install -q diff-biophys==0.2.0 matplotlib
import jax
import jax.numpy as jnp
import matplotlib.pyplot as plt
import numpy as np

plt.rcParams.update({
    "figure.dpi": 120,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "font.size": 12,
})
print("Ready. JAX devices:", jax.devices())

---
## 1 · What is Circular Dichroism?

**Circular dichroism (CD)** exploits the fact that chiral molecules (like proteins)
absorb **left-handed** and **right-handed** circularly polarised light *differently*.

The CD signal at each wavelength is:

$$\Delta\varepsilon(\lambda) = \varepsilon_L(\lambda) - \varepsilon_R(\lambda)$$

The amide bonds (C=O ··· N–H) in the protein backbone are the main chromophores
in the far-UV region (180–260 nm).  The *coupling* between neighbouring amide
transition dipoles produces the characteristic spectral shapes:

| Secondary structure | 222 nm | 208 nm | 193 nm |
|---|---|---|---|
| **α-helix** | negative | negative | **positive** |
| **β-sheet** | negative | — | **positive** (weaker) |
| **Random coil** | — | — | negative |

These signatures are used daily in biochemistry labs to rapidly assess whether
a protein is folded, what its secondary-structure content is, and whether a
mutation or ligand disrupts the fold.

### The physics: coupled oscillators

Each amide bond has a transition dipole moment $\boldsymbol{\mu}_i$.
When two dipoles interact, they form in-phase and out-of-phase combinations
with *different* absorption energies.  The CD signal arises from the
**rotational strength** of these coupled transitions — which depends on
both the *distances* and *orientations* of the dipoles.

`simulate_cd_matrix` implements the full **DeVoe matrix method** differentiably.


---
## 2 · Building a Helix from Scratch

Before we can compute a CD spectrum, we need atomic positions.
We'll use the **NeRF (Natural Extension Reference Frame)** algorithm
to place amide nitrogen atoms along a canonical α-helix.

An ideal α-helix has:
- Rise per residue: 1.5 Å
- Helical radius: 2.3 Å  
- 3.6 residues per turn → 100° rotation per residue


In [ ]:
def build_helix_chromophores(n_residues: int, radius: float = 2.3,
                              rise: float = 1.5) -> tuple:
    """
    Place amide N atoms on an ideal α-helix.

    Returns
    -------
    positions  : (n_residues, 3) amide N Cartesian coordinates
    dipoles    : (n_residues, 3) unit transition-dipole vectors (tangent to helix)
    """
    coords = []
    degrees_per_residue = 100.0    # 360° / 3.6 res/turn

    for i in range(n_residues):
        angle = jnp.deg2rad(i * degrees_per_residue)
        x = radius * jnp.cos(angle)
        y = radius * jnp.sin(angle)
        z = i * rise
        coords.append(jnp.array([x, y, z]))

    positions = jnp.stack(coords)                    # (N, 3)

    # Dipoles: tangent to the helix (forward difference, normalised)
    tangents = jnp.roll(positions, -1, axis=0) - positions
    tangents = tangents.at[-1].set(tangents[-2])     # repeat last
    norms    = jnp.linalg.norm(tangents, axis=-1, keepdims=True)
    dipoles  = tangents / norms                      # unit vectors

    return positions, dipoles


n_res = 12
positions, dipoles = build_helix_chromophores(n_res)
print(f"Built {n_res}-residue helix")
print(f"  Positions shape: {positions.shape}")
print(f"  Dipoles   shape: {dipoles.shape}")
print(f"  Helix spans z = {float(positions[0, 2]):.1f} to {float(positions[-1, 2]):.1f} Å")

In [ ]:
# Visualise the helix
fig = plt.figure(figsize=(7, 6))
ax = fig.add_subplot(111, projection="3d")

# Draw helix backbone
xs, ys, zs = np.array(positions[:, 0]), np.array(positions[:, 1]), np.array(positions[:, 2])
ax.plot(xs, ys, zs, "-o", color="steelblue", lw=2, ms=8, label="Amide N positions")

# Draw transition dipoles
scale = 1.5
for i in range(n_res):
    dx, dy, dz = [float(d) * scale for d in dipoles[i]]
    ax.quiver(xs[i], ys[i], zs[i], dx, dy, dz,
              color="crimson", alpha=0.7, arrow_length_ratio=0.3)

ax.set_xlabel("x (Å)"); ax.set_ylabel("y (Å)"); ax.set_zlabel("z (Å)")
ax.set_title(f"{n_res}-residue α-helix\nblue = amide positions, red = transition dipoles")
plt.tight_layout(); plt.show()

---
## 3 · Computing the CD Spectrum

`simulate_cd_matrix` implements the DeVoe coupled-oscillator model.
The function takes:
- `peptide_positions` — where the chromophores are in 3D space
- `dipole_orientations` — which direction each transition dipole points
- `wavelengths` — the wavelengths at which to evaluate the spectrum

And returns the **molar ellipticity** [θ] in deg·cm²/dmol at each wavelength.


In [ ]:
from diff_biophys.cd.kernels import simulate_cd_matrix

wavelengths = jnp.linspace(180.0, 260.0, 81, dtype=jnp.float32)  # 180–260 nm

cd_helix = simulate_cd_matrix(positions, dipoles, wavelengths,
                               f_osc=0.2, gamma=10.0, lambda_0=190.0)

fig, ax = plt.subplots(figsize=(9, 5))
ax.plot(wavelengths, cd_helix, "steelblue", lw=2.5, label="α-helix (12 residues)")
ax.axhline(0, color="k", lw=0.8)
ax.axvline(222, color="crimson",  lw=1.2, ls="--", alpha=0.6, label="222 nm (helix marker)")
ax.axvline(208, color="darkorange", lw=1.2, ls="--", alpha=0.6, label="208 nm (helix marker)")
ax.axvline(193, color="green",    lw=1.2, ls="--", alpha=0.6, label="193 nm (positive band)")
ax.set_xlabel("Wavelength  [nm]")
ax.set_ylabel("[θ]  (deg cm² dmol⁻¹)")
ax.set_title("CD spectrum of an ideal α-helix")
ax.legend(fontsize=9); plt.tight_layout(); plt.show()

print(f"[θ] at 222 nm: {float(cd_helix[jnp.argmin(jnp.abs(wavelengths - 222))]):.1f}  (should be negative)")
print(f"[θ] at 193 nm: {float(cd_helix[jnp.argmin(jnp.abs(wavelengths - 193))]):.1f}  (should be positive)")

---
## 4 · How Geometry Affects the Spectrum

The CD spectrum is exquisitely sensitive to 3D structure.
Let's gradually **unwind the helix** by increasing the z-rise while keeping the radius fixed,
and watch the spectrum change.

A larger rise-per-residue → more extended → less helical coupling → spectrum changes.


In [ ]:
rises = [1.0, 1.5, 2.0, 2.5, 3.0, 3.5]   # Å per residue
colors = plt.cm.viridis(np.linspace(0, 1, len(rises)))

fig, ax = plt.subplots(figsize=(10, 5))

for rise, color in zip(rises, colors):
    pos_r, dip_r = build_helix_chromophores(n_res, radius=2.3, rise=rise)
    cd_r = simulate_cd_matrix(pos_r, dip_r, wavelengths)
    label = f"rise = {rise:.1f} Å {'← ideal helix' if rise == 1.5 else ''}"
    lw = 3.0 if rise == 1.5 else 1.5
    ax.plot(wavelengths, cd_r, lw=lw, color=color, label=label)

ax.axhline(0, color="k", lw=0.8)
ax.axvline(222, color="grey", lw=1, ls="--", alpha=0.5)
ax.set_xlabel("Wavelength  [nm]")
ax.set_ylabel("[θ]  (deg cm² dmol⁻¹)")
ax.set_title("CD spectrum as the helix is unwound (rise per residue increases)")
ax.legend(fontsize=8, loc="lower right"); plt.tight_layout(); plt.show()
print("Notice: the 222 nm negative band diminishes as the helix extends.")

In [ ]:
# Quantify: [theta] at 222 nm vs. rise
cd_at_222 = []
for rise in np.linspace(0.8, 4.0, 40):
    pos_r, dip_r = build_helix_chromophores(n_res, radius=2.3, rise=rise)
    cd_r = simulate_cd_matrix(pos_r, dip_r, wavelengths)
    idx_222 = int(jnp.argmin(jnp.abs(wavelengths - 222)))
    cd_at_222.append(float(cd_r[idx_222]))

fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(np.linspace(0.8, 4.0, 40), cd_at_222, "steelblue", lw=2.5, marker="o", ms=4)
ax.axvline(1.5, color="crimson", lw=1.5, ls="--", label="Ideal α-helix (1.5 Å/res)")
ax.axhline(0, color="k", lw=0.6)
ax.set_xlabel("Rise per residue  [Å]")
ax.set_ylabel("[θ]₂₂₂  (deg cm² dmol⁻¹)")
ax.set_title("[θ] at 222 nm vs. helical rise — a structural ruler")
ax.legend(); plt.tight_layout(); plt.show()

---
## 5 · The Gradient of the CD Spectrum

Here is the most powerful part: `simulate_cd_matrix` is **fully differentiable**.

We can ask: *"At 222 nm, which amide position, if moved, would most change the CD signal?"*

This is $\frac{\partial [\theta](222\,\text{nm})}{\partial \mathbf{r}_i}$ — the gradient
of the spectrum at a specific wavelength with respect to each chromophore's position.

This is the basis of CD-driven structure refinement.


In [ ]:
idx_222 = int(jnp.argmin(jnp.abs(wavelengths - 222)))

def cd_at_222nm(pos):
    """CD signal at 222 nm as a function of chromophore positions."""
    cd = simulate_cd_matrix(pos, dipoles, wavelengths)
    return cd[idx_222]

# Gradient: shape (12, 3) — one 3D vector per chromophore
grad_fn  = jax.grad(cd_at_222nm)
grad_pos = grad_fn(positions)

print("Gradient shape:", grad_pos.shape)
print("Largest gradient magnitude at residue:",
      int(jnp.argmax(jnp.linalg.norm(grad_pos, axis=-1))))

# Visualise the gradient magnitudes
grad_mags = jnp.linalg.norm(grad_pos, axis=-1)

fig, ax = plt.subplots(figsize=(8, 4))
bars = ax.bar(range(1, n_res + 1), grad_mags, color="steelblue", edgecolor="white")
ax.set_xlabel("Residue"); ax.set_ylabel("|d[θ]₂₂₂ / dr|")
ax.set_title("Gradient magnitude: which amide positions most influence [θ] at 222 nm?")
plt.tight_layout(); plt.show()

In [ ]:
# Visualise gradient vectors in 3D
fig = plt.figure(figsize=(8, 7))
ax = fig.add_subplot(111, projection="3d")

xs = np.array(positions[:, 0])
ys = np.array(positions[:, 1])
zs = np.array(positions[:, 2])

ax.plot(xs, ys, zs, "-o", color="steelblue", lw=1.5, ms=7, alpha=0.6)

# Scale gradient arrows for visibility
scale = 8.0 / float(jnp.max(grad_mags) + 1e-8)
for i in range(n_res):
    gx, gy, gz = [float(g) * scale for g in grad_pos[i]]
    intensity = float(grad_mags[i]) / float(jnp.max(grad_mags))
    ax.quiver(xs[i], ys[i], zs[i], gx, gy, gz,
              color=plt.cm.hot(intensity), arrow_length_ratio=0.4, lw=2)

ax.set_xlabel("x (Å)"); ax.set_ylabel("y (Å)"); ax.set_zlabel("z (Å)")
ax.set_title("d[θ]₂₂₂/dr — gradient of 222 nm signal w.r.t. amide positions\n"
             "(hot = large gradient, blue = small)")
plt.tight_layout(); plt.show()
print("\n💡 Moving high-gradient residues will change the 222 nm signal most.")

In [ ]:
# --- Mini refinement: drive [theta] at 222nm more negative ---
import optax

# Starting from the ideal helix; we want to maximise |[theta]_222|
# equivalently: minimise cd_at_222nm (make it more negative)
def loss(pos):
    return cd_at_222nm(pos)   # we want this to go down (more negative)

opt = optax.adam(learning_rate=0.05)
pos = positions  # start from ideal helix
state = opt.init(pos)
grad_fn_jit = jax.jit(jax.value_and_grad(loss))

cd_trajectory = []
for step in range(60):
    val, grads = grad_fn_jit(pos)
    cd_trajectory.append(float(val))
    updates, state = opt.update(grads, state)
    pos = optax.apply_updates(pos, updates)

fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(cd_trajectory, "crimson", lw=2)
ax.set_xlabel("Optimisation step")
ax.set_ylabel("[θ]₂₂₂  (deg cm² dmol⁻¹)")
ax.set_title("Driving [θ] at 222 nm more negative via gradient descent")
plt.tight_layout(); plt.show()
print(f"[θ]₂₂₂ :  {cd_trajectory[0]:.1f}  →  {cd_trajectory[-1]:.1f}")
print("\n✅ CD-driven gradient descent moved the chromophores to strengthen the helical signal!")

---
## 6 · Summary

| Concept | Key Point |
|---|---|
| CD signal at 222 nm | Negative = α-helix character; magnitude reports helical content |
| `simulate_cd_matrix` | Full DeVoe coupled-oscillator model, fully differentiable in JAX |
| `jax.grad(cd_at_222nm)(positions)` | Tells you which chromophores most influence the signal |
| Gradient descent on CD | Moves chromophore positions to match a target spectrum |

### The bigger picture

You have now seen three completely different physical experiments —
**SAXS** (X-ray scattering), **NMR** (magnetic resonance), **CD** (polarised light) —
all implemented as differentiable functions in `diff-biophys`.

Because they all produce JAX arrays, you can **combine them**:

```python
total_loss = saxs_chi2(coords) + nmr_mse(phi, psi) + cd_mse(chromophore_pos)
grads = jax.grad(total_loss)(all_params)
```

This is multi-experiment structure refinement — the frontier of the field — in one line.

### Further reading
- Greenfield 2006 — *Using circular dichroism spectra to estimate protein secondary structure* — Nature Protocols
- Kelly et al. 2005 — *How to study proteins by circular dichroism* — BBA
